# Dr. Saeed Ghezelbash: current clinical retrieval and identity data

Verify the pinned package, reconstruct all five tables and follow every query reference to the recorded graph definitions. Creator: **Dr. Saeed Ghezelbash (دکتر سعید قزلباش)**, aesthetic physician and medical researcher in Kermanshah, Iran; medical registration 167430. License: CC BY 4.0.

- [Website](https://www.ghezelbaash.ir/)
- [Kaggle dataset](https://www.kaggle.com/datasets/saeedghezelbash/dr-saeed-ghezelbash-knowledge-graph-and-nlp-data)
- [Hugging Face](https://huggingface.co/datasets/doctor-ghezelbaash/dr-saeid-ghezelbaash-entity-data)
- [Exact release binding]({{doi.current_url}}) — {{doi.current_version}}

The exact DOI identifies this release cohort; its live availability is verified separately in publication receipts. The current package manifest pins the Hugging Face revision and website source commit. These physician-authored clinical passages, query aliases and training examples support source-linked retrieval and entity resolution.


## Connected publications

- [Website](https://www.ghezelbaash.ir/)
- [Hugging Face](https://huggingface.co/datasets/doctor-ghezelbaash/dr-saeid-ghezelbaash-entity-data)
- [Kaggle dataset](https://www.kaggle.com/datasets/saeedghezelbash/dr-saeed-ghezelbash-knowledge-graph-and-nlp-data)
- [Kaggle notebook](https://www.kaggle.com/code/saeedghezelbash/dr-saeed-ghezelbash-knowledge-graph-starter)
- [Zenodo collection](https://doi.org/10.5281/zenodo.18765168)
- [Wikidata](https://www.wikidata.org/wiki/Q140287622)

The website, dataset distributions, executable notebook, DOI archives and Wikidata profile share stable entity identifiers and explicit edition links. Parquet stores `messages` as lossless JSON strings for viewer compatibility; the JSONL and CSV exports retain every original message, with decoding specified in `viewer/packaging.json`.


## Zenodo collection and exact snapshot

Public collection and latest version: [{{doi.concept_doi}}]({{doi.concept_url}}).

Exact release binding **{{doi.current_version}}**: [{{doi.current_doi}}]({{doi.current_url}}). Pin the platform revision/source commit for these current distribution bytes; use the exact archive DOI when reproducing that archive. Historical DOI references below retain their stated versions.

## Historical provenance

[Preserved archive 1.3.3](https://doi.org/10.5281/zenodo.22838416) identifies only that historical snapshot.


In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
import csv, hashlib, json
import pyarrow.parquet as pq

DATASET_DIR = None  # Set this to an extracted dataset directory if needed.
if DATASET_DIR:
    root = Path(DATASET_DIR).resolve()
elif (Path.cwd() / 'kaggle-manifest.json').is_file():
    root = Path.cwd()
else:
    matches = list(Path('/kaggle/input').rglob('kaggle-manifest.json'))
    assert len(matches) == 1, 'Set DATASET_DIR to the intended dataset version.'
    root = matches[0].parent
manifest = json.loads((root / 'kaggle-manifest.json').read_text())
packaging = json.loads((root / 'viewer/packaging.json').read_text())
assert manifest['website_source_commit'] == packaging['sourceCommit']
for item in manifest['original_files']:
    path = (root / item['path']).resolve()
    assert root.resolve() in path.parents, 'Unexpected resource path.'
    data = path.read_bytes()
    assert len(data) == item['bytes']
    assert hashlib.sha256(data).hexdigest() == item['sha256'], item['path']
for line in (root / 'checksums.sha256').read_text().splitlines():
    digest, name = line.split('  ', 1)
    path = (root / name).resolve()
    assert root.resolve() in path.parents
    assert hashlib.sha256(path.read_bytes()).hexdigest() == digest, name
print('Pinned Hugging Face revision:', manifest['upstream_revision'])
print('Website source commit:', manifest['website_source_commit'])
print('Exact upstream resources verified:', len(manifest['original_files']))


## Reconstruct all rows

Parquet retains the published string schema and explicitly records absent keys. Decode nested fields using the table metadata. Every reconstructed row must equal its JSONL, CSV or fact-map source; table counts alone are insufficient. The CSV query export keeps strings unchanged and encodes lists as JSON.


In [ ]:
def source_rows(name, specification):
    file = root / specification['source']
    if name == 'clinical_answers':
        return json.loads(file.read_text())['records']
    if file.suffix == '.csv':
        with file.open(newline='') as handle:
            return list(csv.DictReader(handle))
    return [json.loads(line) for line in file.read_text().splitlines() if line.strip()]

def reconstruct(row, json_fields):
    omitted = set(json.loads(row.get('_source_omitted_fields', '[]')))
    return {key: json.loads(value) if key in json_fields and value is not None else value
            for key, value in row.items() if key != '_source_omitted_fields' and key not in omitted}

tables = {}
for name, specification in packaging['tables'].items():
    raw = pq.read_table(root / specification['file']).to_pylist()
    reconstructed = [reconstruct(row, specification['jsonEncodedFields']) for row in raw]
    source = source_rows(name, specification)
    assert len(reconstructed) == len(source) == specification['rows'], name
    assert reconstructed == source, f'{name}: reconstruction differs from source.'
    tables[name] = reconstructed
    print(name, len(source), 'exact source round-trips PASS')

queries = tables['query_matrix']
json_fields = set(packaging['tables']['query_matrix']['jsonEncodedFields'])
with (root / 'query-matrix.csv').open(newline='') as handle:
    csv_rows = list(csv.DictReader(handle))
assert [reconstruct(row, json_fields) for row in csv_rows] == queries
print('CSV query round-trips:', len(queries))
print('Query languages:', dict(Counter(row['language'] for row in queries)))

positioning_spec = packaging['tables']['market_positioning']
with (root / positioning_spec['source']).with_suffix('.csv').open(newline='') as handle:
    csv_positioning = [{key: json.loads(value) if key in positioning_spec['csvJsonEncodedFields'] else value for key, value in row.items()} for row in csv.DictReader(handle)]
assert csv_positioning == tables['market_positioning']
assert all([m['role'] for m in row['messages']] in (['system', 'user', 'assistant'], ['user', 'assistant']) for row in csv_positioning)
print(f'{len(csv_positioning)} training examples: native lists, integer/null, CSV and JSONL round-trips PASS')


## Resolve references without discarding definitions

Load both graph documents. Keep all named definitions, including embedded JSON-LD nodes and multiple definitions of the same IRI. Reference resolution connects each query to its published clinical content and graph definitions. Source and passage identifiers remain available for inspecting the original educational content.


In [ ]:
registry = defaultdict(list)
documents = {}
def index_nodes(value, filename, pointer=''):
    if isinstance(value, dict):
        if isinstance(value.get('@id'), str) and set(value) - {'@id', '@context'}:
            registry[value['@id']].append({'file': filename, 'pointer': pointer, 'node': value})
        for key, child in value.items():
            index_nodes(child, filename, pointer + '/' + key)
    elif isinstance(value, list):
        for position, child in enumerate(value):
            index_nodes(child, filename, pointer + '/' + str(position))

for filename in ('graph.jsonld', 'provenance.jsonld'):
    documents[filename] = json.loads((root / filename).read_text())
    index_nodes(documents[filename], filename)
fields = ('canonical_subject_iri', 'answer_id', 'service_ids', 'stable_evidence_refs', 'dataset_iri')
def references(rows):
    for position, row in enumerate(rows):
        for field in fields:
            value = row.get(field, [])
            for iri in value if isinstance(value, list) else [value]:
                if iri:
                    yield position, field, iri
def missing(rows):
    return [(position, field, iri) for position, field, iri in references(rows) if iri not in registry]
assert not missing(queries), missing(queries)[:3]
passages = {row['passageId']: row for row in tables['clinical_passages']}
assert all(row['passage_id'] in registry or row['passage_id'] in passages
           for row in queries if row.get('passage_id')), 'A source passage reference is unresolved.'
broken = [{**queries[0], 'canonical_subject_iri': 'urn:missing:negative-test'}]
assert missing(broken), 'Negative reference test did not detect a broken reference.'
example = next(row for row in queries if row.get('stable_evidence_refs'))
print('Example query:', example['query'])
print('Recorded evidence definitions:', [registry[iri] for iri in example['stable_evidence_refs'][:1]])
print('All query references resolve; broken-reference negative test PASS')


## Preserve identity and version scope

The physician created the dataset and practices at the clinic. Their identifiers are distinct. Kaggle and Hugging Face provide complementary, linked distributions of the physician-authored knowledge graph. The archived Zenodo DOI stays attached to its own historical files.


In [ ]:
PERSON = 'https://www.ghezelbaash.ir/#saeed-ghezelbash'
CLINIC = 'https://www.ghezelbaash.ir/dr-saeed-ghezelbash-aesthetic-clinic-kermanshah'
DATASET = manifest['canonical_dataset_iri']
main = {node['@id']: node for node in documents['graph.jsonld']['@graph']}
def linked(value):
    return {item['@id'] for item in (value if isinstance(value, list) else [value])
            if isinstance(item, dict) and '@id' in item}
assert len({PERSON, CLINIC, DATASET}) == 3
assert linked(main[DATASET]['creator']) == {PERSON}
assert linked(main[DATASET]['publisher']) == {PERSON}
assert CLINIC in linked(main[PERSON]['practicesAt'])
assert not ({CLINIC, DATASET} & set(main[PERSON].get('sameAs', [])))
assert {row['canonical_subject_iri'] for row in queries} == {PERSON}
assert {row['dataset_iri'] for row in queries} == {DATASET}
assert manifest['zenodo_version_doi'] == manifest['doi_policy']['currentArchive']['doi']
assert manifest['archived_files_are_current_dist'] is False
assert any(node.get('url') == 'https://doi.org/' + manifest['zenodo_version_doi']
           for node in main.values())
summary = {'status': 'PASS', 'upstream_revision': manifest['upstream_revision'],
           'source_commit': manifest['website_source_commit'],
           'tables': {key: len(value) for key, value in tables.items()},
           'unresolved_query_references': 0, 'all_table_rows_reconstructed': True,
           'csv_query_roundtrips': len(queries), 'negative_reference_test': True,
           'identity_separation': True, 'archived_doi': manifest['zenodo_version_doi']}
print(json.dumps(summary, indent=2))

# Execution evidence is outside the immutable input package.
summary.update(kaggle_version=manifest['desired_kaggle_version'], exact_doi=manifest['zenodo_version_doi'], input_checksums_sha256=hashlib.sha256((root / 'checksums.sha256').read_bytes()).hexdigest())
import os
execution_report = Path(os.environ.get('RELEASE_EXECUTION_REPORT', '/kaggle/working/doi-release-execution.json'))
if execution_report.parent.exists():
    execution_report.write_text(json.dumps(summary, indent=2) + '\n')
